# EEG · 뇌파에서 눈 깜빡임 흔적 찾아보기 (합성 데이터 예제)

이 노트북은 **연습용 합성 데이터**를 사용합니다. 실제 사람의 뇌파가 아니며, 눈 깜빡임과 눈 움직임이 언제 들어갔는지 정답이 기록되어 있습니다.

먼저 같은 폴더에서 데이터를 만듭니다 (몇 초).

```bash
python generate_synthetic_eeg.py
```

`data/ground_truth.json`에 정답이 들어 있습니다. **마지막 단계 전까지 열어 보지 마세요.**

In [ ]:
import json
import matplotlib.pyplot as plt
import mne
import numpy as np
from scipy.signal import find_peaks

raw = mne.io.read_raw_fif("data/eeg_raw.fif", verbose="error")
data = raw.get_data() * 1e6          # MNE는 볼트(V)로 읽으므로 마이크로볼트(µV)로 바꿈
sfreq, chs = raw.info["sfreq"], raw.ch_names
time = np.arange(data.shape[1]) / sfreq
print(f"채널 {len(chs)}개: {chs}")
print(f"초당 {sfreq:.0f}번 측정, 길이 {time[-1]:.0f}초")

## 1. 신호 그려 보기

EEG는 머리 표면 여러 위치(채널)에서 잰 전압입니다. 이름 앞 글자가 위치를 뜻합니다: Fp=이마, F=앞쪽, C=가운데, T=옆머리, P=정수리 뒤, O=뒤통수. 처음 20초를 그려 봅니다.

In [ ]:
def plot_channels(start, length, channels=chs, spacing=150, marks=()):
    s, e = int(start * sfreq), int((start + length) * sfreq)
    fig, ax = plt.subplots(figsize=(13, 0.45 * len(channels) + 1))
    for k, ch in enumerate(channels):
        ax.plot(time[s:e], data[chs.index(ch), s:e] - spacing * k, lw=0.7, color="k")
    for m in marks:
        if start <= m <= start + length:
            ax.axvline(m, color="r", alpha=0.3)
    ax.set_yticks([-spacing * k for k in range(len(channels))], channels)
    ax.set_xlabel("time (s)")
    ax.set_title(f"vertical spacing = {spacing} µV")
    return ax

plot_channels(0, 20);

**생각해 보기:** 가끔 크게 솟는 파형이 보이나요? 어느 채널에서 가장 크고, 뒤쪽 채널로 갈수록 어떻게 변하나요?

## 2. 큰 파형 자동으로 찾기

이마 채널(Fp1, Fp2)의 평균을 느린 성분만 남기고(10 Hz 저역 통과), 기준보다 높이 솟은 정점을 찾습니다.

In [ ]:
frontal = data[[chs.index("Fp1"), chs.index("Fp2")]].mean(axis=0)
frontal_slow = mne.filter.filter_data(frontal, sfreq, None, 10.0, verbose="error")
threshold = 80   # µV
peaks, _ = find_peaks(frontal_slow, height=threshold, distance=int(0.3 * sfreq))
detected = peaks / sfreq
print(f"{threshold} µV를 넘는 정점 {len(detected)}개, 분당 {len(detected) / (time[-1] / 60):.1f}회")
plot_channels(0, 20, marks=detected);

## 3. 파형이 어디서 왔을까? 두피 분포 보기

찾은 시점마다 모든 채널의 값을 모아 평균하면, 그 파형이 머리의 어느 쪽에서 크게 나타나는지 지도로 볼 수 있습니다.

In [ ]:
amplitude = np.mean([data[:, p] - data[:, p - int(0.15 * sfreq)] for p in peaks], axis=0)
fig, ax = plt.subplots(figsize=(4, 4))
mne.viz.plot_topomap(amplitude, raw.info, axes=ax, show=False, cmap="RdBu_r")
ax.set_title("mean change at detected peaks (µV)");
for ch in ["Fp1", "Fz", "Cz", "Pz", "O1"]:
    print(f"{ch}: {amplitude[chs.index(ch)]:+.0f} µV")

**생각해 보기:** 이마 앞쪽에서 가장 크고 뒤로 갈수록 작아져 부호까지 바뀝니다. 뇌 안쪽보다 **눈 쪽에서 온 신호**라고 볼 근거가 되나요?

## 4. 다른 종류의 큰 신호: 눈 좌우 움직임

눈을 좌우로 움직이면 눈 양옆의 F7과 F8에 **부호가 반대인** 계단 모양 신호가 생깁니다. F7 − F8을 그려 봅니다.

In [ ]:
f7_f8 = data[chs.index("F7")] - data[chs.index("F8")]
f7_f8_slow = mne.filter.filter_data(f7_f8, sfreq, None, 5.0, verbose="error")
fig, ax = plt.subplots(figsize=(13, 3))
ax.plot(time, f7_f8_slow, lw=0.6, color="k")
ax.set_xlabel("time (s)"); ax.set_ylabel("F7 - F8 (µV)")
ax.set_title("horizontal eye movement proxy");
start = min(time[np.argmax(np.abs(f7_f8_slow))] - 3, time[-1] - 8)
plot_channels(start, 8, channels=["Fp1", "Fp2", "F7", "F8", "Cz", "O1"], spacing=250);

**생각해 보기:** 2단계의 방법은 이 좌우 움직임을 눈 깜빡임으로 잘못 세었을까요? F7과 F8만 보고 기준을 정했다면 어떻게 되었을까요?

## 5. 정답과 비교

이제 `ground_truth.json`을 엽니다. 찾은 시점이 정답과 0.1초 안에 있으면 맞힌 것으로 봅니다.

In [ ]:
truth = json.load(open("data/ground_truth.json", encoding="utf-8"))
true_blinks = np.array([b["peak_s"] for b in truth["blinks"]])
hit = np.array([np.min(np.abs(true_blinks - d)) <= 0.1 for d in detected])
found = np.array([np.min(np.abs(detected - t)) <= 0.1 for t in true_blinks]) if len(detected) else np.zeros(len(true_blinks), bool)
print(f"정답 눈 깜빡임 {len(true_blinks)}회 중 {found.sum()}회 찾음")
print(f"찾은 {len(detected)}개 중 틀린 것 {(~hit).sum()}개")
missed = [b["fp_amplitude_uv"] for b, f in zip(truth["blinks"], found) if not f]
print("놓친 깜빡임의 크기 (µV):", missed)
print("좌우 눈 움직임:", len(truth["saccades"]), "회 —", [s["onset_s"] for s in truth["saccades"]])

## 설명해 보세요

1. 놓친 깜빡임이 있다면 왜 놓쳤을까요? `threshold`를 50이나 120으로 바꾸면 무엇이 달라지나요?
2. 눈 깜빡임과 좌우 눈 움직임을 구분하는 근거를 두 가지 적어 보세요 (모양, 위치, 부호).
3. 실제 기록이라면 이 신호가 눈 깜빡임이라고 확신하려면 어떤 정보가 더 필요할까요? (힌트: 눈 전용 전극 EOG, 영상 기록)

**다른 데이터로 반복 (과제 5):** `python generate_synthetic_eeg.py --seed 7 --out data_seed7`로 다른 데이터를 만들고, 이 노트북에서 `"data/`로 시작하는 경로 두 곳을 `"data_seed7/`로 바꿔 같은 방법이 얼마나 잘 작동하는지 확인해 보세요.